In [1]:
from torch_geometric.loader import DataLoader
from torch_geometric.nn import SAGEConv, global_mean_pool , GlobalAttention
from torch.nn import MultiheadAttention

from torch_geometric.transforms import ToUndirected
from torch_geometric.datasets import UPFD
import os.path as osp
import torch_geometric
import numpy as np
import torch.nn as nn
import torch

In [2]:
dataset = UPFD(root="data/UPFD", name="politifact", feature="bert", split="train")
train_loader = DataLoader(dataset, batch_size=128, shuffle=True)
test_dataset = UPFD(root="data/UPFD", name="politifact", feature="bert", split="test")
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [3]:
class GNN(nn.Module):
    def __init__(self,input,output):
        super().__init__()
        self.conv1 = SAGEConv(input,32)
        self.conv2 = SAGEConv(32,32)
        self.gate = nn.Linear(32,1)
        self.attention = GlobalAttention(gate_nn=self.gate)
        self.dropout = nn.Dropout(0.1)
        self.last = torch.nn.Linear(32,output)
    def forward(self,x,edge_index,batch):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index).relu()
        x = self.attention(x, batch)
        output = self.last(x)
        
        return output

In [4]:
model = GNN(768,1)
print(model)

GNN(
  (conv1): SAGEConv(768, 32, aggr=mean)
  (conv2): SAGEConv(32, 32, aggr=mean)
  (gate): Linear(in_features=32, out_features=1, bias=True)
  (attention): GlobalAttention(gate_nn=Linear(in_features=32, out_features=1, bias=True), nn=None)
  (dropout): Dropout(p=0.1, inplace=False)
  (last): Linear(in_features=32, out_features=1, bias=True)
)


c:\Users\Sarvesh\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch_geometric\deprecation.py:26: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  warnings.warn(out)


In [5]:
optimizer = torch.optim.Adam(model.parameters(),lr = 0.003)
loss_fn = torch.nn.BCEWithLogitsLoss()
def train():
    loss =0
    model.train()
    for data in train_loader:  
        optimizer.zero_grad()  
        out = model(data.x, data.edge_index, data.batch)
        #out.reshape((out.shape,1))
        
        
        #break
        y_vals = data.y.float()
        y_vals=torch.reshape(y_vals,(y_vals.shape[0],1))   
        #print(y_vals)
        #print(data.y)
        curr_loss = loss_fn(out, y_vals)
        
        curr_loss.backward() 
        optimizer.step()  
        loss += curr_loss.item() * data.num_graphs
        return loss/(len(train_loader.dataset))

for epoch in range (1,60):
    loss = train()
    print(f'Epoch: {epoch:02d}, Loss: {loss:.4f}')



Epoch: 01, Loss: 0.7081
Epoch: 02, Loss: 0.6771
Epoch: 03, Loss: 0.6637
Epoch: 04, Loss: 0.6427
Epoch: 05, Loss: 0.6235
Epoch: 06, Loss: 0.6136
Epoch: 07, Loss: 0.5665
Epoch: 08, Loss: 0.5479
Epoch: 09, Loss: 0.5175
Epoch: 10, Loss: 0.4877
Epoch: 11, Loss: 0.4557
Epoch: 12, Loss: 0.4123
Epoch: 13, Loss: 0.3795
Epoch: 14, Loss: 0.3613
Epoch: 15, Loss: 0.3091
Epoch: 16, Loss: 0.2773
Epoch: 17, Loss: 0.2321
Epoch: 18, Loss: 0.1992
Epoch: 19, Loss: 0.1588
Epoch: 20, Loss: 0.1368
Epoch: 21, Loss: 0.1365
Epoch: 22, Loss: 0.1323
Epoch: 23, Loss: 0.0820
Epoch: 24, Loss: 0.0835
Epoch: 25, Loss: 0.0702
Epoch: 26, Loss: 0.0515
Epoch: 27, Loss: 0.0502
Epoch: 28, Loss: 0.0382
Epoch: 29, Loss: 0.0278
Epoch: 30, Loss: 0.0218
Epoch: 31, Loss: 0.0179
Epoch: 32, Loss: 0.0188
Epoch: 33, Loss: 0.0112
Epoch: 34, Loss: 0.0220
Epoch: 35, Loss: 0.0091
Epoch: 36, Loss: 0.0091
Epoch: 37, Loss: 0.0094
Epoch: 38, Loss: 0.0030
Epoch: 39, Loss: 0.0031
Epoch: 40, Loss: 0.0063
Epoch: 41, Loss: 0.0058
Epoch: 42, Loss:

In [6]:
@torch.no_grad
def test(loader):
    model.eval()
    correct = 0
    
    for data in loader:
        out = model(data.x, data.edge_index, data.batch)
        pred = (torch.sigmoid(out) >= 0.5).squeeze().long()
        correct += int((pred == data.y).sum())
        #print(torch.sigmoid(out))
            
    return correct / len(loader.dataset)

In [7]:
test(test_loader)

0.832579185520362